# Training an LLM on Multiple Choice Questions with Tinker + Verifiers

This notebook demonstrates how to train a language model using Reinforcement Learning to answer multiple-choice questions. We use:

- **Tinker**: Post-training framework with RL capabilities
- **Verifiers**: RL environment library for LLMs
- **Custom MCQ Environment**: A simple environment with 25 multiple-choice questions

## Overview

The model learns to:
1. Read a multiple-choice question with options A-D
2. Generate an answer in XML format: `<answer>X</answer>`
3. Receive a reward of 1.0 for correct answers, 0.0 for incorrect

Through RL training, the model improves its accuracy on these questions.

## Setup and Installation

First, ensure you have the required dependencies installed.

In [1]:
# Install dependencies if needed
# !pip install tinker verifiers datasets python-dotenv openai

In [2]:
import sys
import os
import asyncio
from pathlib import Path
from datasets import Dataset

# Add the project root to the Python path
project_root = Path.cwd().parent  # rl-pi-test is in project root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Monkey-patch Dataset to add repeat method that verifiers expects
def _dataset_repeat(self, n: int) -> Dataset:
    """Repeat each example in the dataset n times."""
    repeated_data = []
    for item in self:
        for _ in range(n):
            repeated_data.append(dict(item))
    return Dataset.from_list(repeated_data)

if not hasattr(Dataset, 'repeat'):
    Dataset.repeat = _dataset_repeat
    print("Added repeat() method to Dataset class")

print(f"Project root: {project_root}")

Added repeat() method to Dataset class
Project root: /home/theyashwanthsai/Dev/PostTrain-Lab


## 1. Test the MCQ Environment

Before training, let's load the environment and inspect the dataset.

In [3]:
# Import the MCQ environment
from mcq_env import load_environment

# Load the environment
env = load_environment()
dataset = env.get_dataset()

print(f"Dataset size: {len(dataset)}")
print(f"\nFirst question:")
print(dataset[0]['question'])
print(f"\nCorrect answer: {dataset[0]['answer']}")
print(f"Explanation: {dataset[0]['explanation']}")

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Dataset size: 25

First question:
What is the capital of France?

A. London
B. Berlin
C. Paris
D. Madrid


Correct answer: C
Explanation: Paris is the capital and largest city of France.


## 2. Evaluate Base Model Performance

Before training, let's see how well the base model performs on this task.

In [4]:
import tinker
from tinker_cookbook import renderers, model_info
from tinker_cookbook.tokenizer_utils import get_tokenizer
from tinker_cookbook.recipes.verifiers_rl.tinker_openai import TinkerAsyncOpenAIClient
from verifiers.clients import OpenAIChatCompletionsClient

# Model configuration
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Initialize Tinker service
service = tinker.ServiceClient()
sampling = service.create_sampling_client(base_model=MODEL_NAME)

# Initialize tokenizer and renderer
tokenizer = get_tokenizer(MODEL_NAME)
renderer_name = model_info.get_recommended_renderer_name(MODEL_NAME)
renderer = renderers.get_renderer(renderer_name, tokenizer)

# Create OpenAI-compatible client backed by Tinker
tinker_client = TinkerAsyncOpenAIClient(sampling, renderer, tokenizer)

# Wrap in verifiers Client for compatibility with verifiers 0.1.11+
client = OpenAIChatCompletionsClient(tinker_client)

print(f"Model: {MODEL_NAME}")
print(f"Renderer: {renderer_name}")
print("Client initialized successfully!")

Model: Qwen/Qwen3-4B-Instruct-2507
Renderer: qwen3_instruct
Client initialized successfully!


In [5]:
# Run baseline evaluation
import verifiers as vf
import time
import numpy as np

print("Running baseline evaluation on base model...")
print("This may take a few minutes...\n")

start_time = time.time()
results = env.evaluate_sync(
    client=client,
    model="tinker",
    num_examples=10,  # Evaluate on 10 questions
    rollouts_per_example=3,  # 3 attempts per question
    max_concurrent=16,
    sampling_args={
        "max_tokens": 50,
        "temperature": 0.7,
    }
)
end_time = time.time()

# Extract per-rollout data from results
outputs = results["outputs"]
rewards = [o["reward"] for o in outputs]

print(f"Evaluation completed in {end_time - start_time:.2f} seconds")
print("=" * 60)
print(f"Average Reward: {sum(rewards) / len(rewards):.3f}")
print(f"Std Dev: {np.std(rewards):.3f}")
print(f"Total Evaluations: {len(rewards)}")

# Show per-rollout breakdown
rollouts_per_example = 3
n = len(rewards) // rollouts_per_example
print("\nPer-rollout breakdown:")
for i in range(rollouts_per_example):
    trials = [round(rewards[(i * n) + j], 3) for j in range(n)]
    print(f"Rollout {i + 1}: {trials}")

eval_dataset is not set, falling back to train dataset


Running baseline evaluation on base model...
This may take a few minutes...



Processing 10 groups (30 total rollouts): 100%|██████████| 10/10 [00:02<00:00,  4.23it/s, reward=0.9] 

Evaluation completed in 2.38 seconds
Average Reward: 0.900
Std Dev: 0.300
Total Evaluations: 30

Per-rollout breakdown:
Rollout 1: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Rollout 2: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Rollout 3: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]


## 3. Inspect Sample Outputs

Let's see what the base model's responses look like.

In [6]:
from verifiers.utils.message_utils import messages_to_printable

# Display first 3 examples
for i in range(min(3, len(outputs))):
    o = outputs[i]
    print(f"\n{'=' * 60}")
    print(f"Example {i + 1}:")
    print("=" * 60)
    
    if o.get("prompt"):
        print("\n[PROMPT]")
        print(messages_to_printable(o["prompt"]))
    
    if o.get("completion"):
        print("\n[COMPLETION]")
        print(messages_to_printable(o["completion"]))
    
    print(f"\n[REWARD]: {o['reward']:.3f}")


Example 1:

[PROMPT]
[SystemMessage(role='system', content='\nYou are a helpful assistant that answers multiple choice questions.\n\nYou will be given a question and options. Respond with the correct option letter only out of 4 Letters, in XML format:\n\n<answer>option</answer>\n'), UserMessage(role='user', content='What is the capital of France?\n\nA. London\nB. Berlin\nC. Paris\nD. Madrid\n')]

[COMPLETION]
[AssistantMessage(role='assistant', content='<answer>C</answer>', reasoning_content=None, thinking_blocks=None, tool_calls=None)]

[REWARD]: 1.000

Example 2:

[PROMPT]
[SystemMessage(role='system', content='\nYou are a helpful assistant that answers multiple choice questions.\n\nYou will be given a question and options. Respond with the correct option letter only out of 4 Letters, in XML format:\n\n<answer>option</answer>\n'), UserMessage(role='user', content='What is the capital of France?\n\nA. London\nB. Berlin\nC. Paris\nD. Madrid\n')]

[COMPLETION]
[AssistantMessage(role='a

## 4. Configure RL Training

Now let's set up the training configuration. We'll use LoRA for efficient fine-tuning.

In [7]:
from datetime import datetime

# Training configuration
TRAINING_CONFIG = {
    # Model settings
    "model_name": "Qwen/Qwen3-4B-Instruct-2507",
    "lora_rank": 32,
    
    # Environment settings
    "vf_env_id": "mcq",  # Custom environment
    "dataset_n": -1,  # Use all examples
    
    # Training hyperparameters
    "group_size": 8,  # 8 rollouts per question
    "groups_per_batch": 32,  # 32 questions per batch
    "num_substeps": 1,
    "learning_rate": 1e-5,
    "max_tokens": 50,  # Short answers needed
    "temperature": 0.8,
    "kl_penalty_coef": 0.0,
    "max_concurrent_generation": 16,
    "max_concurrent_scoring": 16,
    
    # Logging
    "eval_every": 0,
    "save_every": 10,
    "wandb_project": None,  # Set to enable W&B logging
}

# Generate run name
model_short = TRAINING_CONFIG["model_name"].replace("/", "-")
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_name = f"mcq_{model_short}_lr{TRAINING_CONFIG['learning_rate']}_{timestamp}"
log_path = f"/tmp/tinker-examples/mcq/{run_name}"

print(f"Run name: {run_name}")
print(f"Log path: {log_path}")
print(f"\nTotal rollouts per batch: {TRAINING_CONFIG['group_size'] * TRAINING_CONFIG['groups_per_batch']}")

Run name: mcq_Qwen-Qwen3-4B-Instruct-2507_lr1e-05_20260401-212914
Log path: /tmp/tinker-examples/mcq/mcq_Qwen-Qwen3-4B-Instruct-2507_lr1e-05_20260401-212914

Total rollouts per batch: 256


## 5. Train the Model

Now we'll run the RL training loop. This will:
1. Load the base model with LoRA adapters
2. Generate rollouts on MCQ questions
3. Update the policy based on rewards
4. Save checkpoints periodically

**Note**: This is a simplified in-notebook training. For production use, run the training script directly:
```bash
python -m tinker_cookbook.recipes.verifiers_rl.train \
    vf_env_id=mcq \
    learning_rate=1e-5 \
    groups_per_batch=32 \
    group_size=8
```

In [8]:
import asyncio
from tinker_cookbook.recipes.verifiers_rl.verifiers_env import (
    VerifiersEnvGroupBuilder,
    VerifiersRLDatasetBuilder,
    convert_states_to_trajectory_group,
    set_vf_env,
)
from tinker_cookbook.rl import train
from tinker_cookbook.rl.types import EnvGroupBuilder, TrajectoryGroup
from tinker_cookbook.completers import TinkerTokenCompleter, TokenCompleter
from verifiers.clients import resolve_client
from typing import cast

# Initialize shared resources
shared_client = None
shared_renderer = None
local_tokenizer = None

async def custom_do_group_rollout(
    builder: EnvGroupBuilder, policy: TokenCompleter
) -> TrajectoryGroup:
    """Custom rollout function for Verifiers environments.
    
    Calls rollout + score_group directly instead of run_group,
    because run_group converts States to RolloutOutput which strips
    the trajectory/token data that convert_states_to_trajectory_group needs.
    """
    global shared_client, shared_renderer, local_tokenizer

    if local_tokenizer is None:
        local_tokenizer = get_tokenizer(TRAINING_CONFIG["model_name"])
    if shared_renderer is None:
        renderer_name = model_info.get_recommended_renderer_name(TRAINING_CONFIG["model_name"])
        shared_renderer = renderers.get_renderer(renderer_name, local_tokenizer)

    sampling_client = cast(TinkerTokenCompleter, policy).sampling_client
    if shared_client is None:
        shared_client = TinkerAsyncOpenAIClient(
            sampling_client, shared_renderer, local_tokenizer
        )
    else:
        shared_client.set_sampling_client(sampling_client)

    vf_builder = cast(VerifiersEnvGroupBuilder, builder)
    rollout_inputs = vf_builder.get_rollout_inputs(TRAINING_CONFIG["group_size"])

    vf_client = OpenAIChatCompletionsClient(shared_client)
    resolved = resolve_client(vf_client)

    sampling_args = {
        "max_tokens": TRAINING_CONFIG["max_tokens"],
        "temperature": TRAINING_CONFIG["temperature"],
    }

    rollout_tasks = [
        vf_builder.vf_env.rollout(inp, resolved, "tinker", sampling_args)
        for inp in rollout_inputs
    ]
    raw_states = await asyncio.gather(*rollout_tasks)

    if vf_builder.vf_env.score_rollouts:
        await vf_builder.vf_env.rubric.score_group(raw_states)
    else:
        await vf_builder.vf_env.rubric.dummy_score_group(raw_states)

    return convert_states_to_trajectory_group(raw_states)

# Override the default rollout function
train.do_group_rollout = custom_do_group_rollout

print("Custom rollout function configured!")

Custom rollout function configured!


In [9]:
# Load and register our custom environment
import verifiers as vf
from mcq_env import load_environment

# Load the environment and set it in context
mcq_env = load_environment()
set_vf_env(mcq_env)

# Create dataset builder
dataset_builder = VerifiersRLDatasetBuilder(
    vf_env_id="mcq",  # This will use our custom environment
    vf_env_args={},
    groups_per_batch=TRAINING_CONFIG["groups_per_batch"],
    dataset_n=TRAINING_CONFIG["dataset_n"],
    dataset_seed=None,
)

print("Dataset builder configured!")

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Dataset builder configured!


In [10]:
# Create training configuration
training_cfg = train.Config(
    learning_rate=TRAINING_CONFIG["learning_rate"],
    dataset_builder=dataset_builder,
    model_name=TRAINING_CONFIG["model_name"],
    max_tokens=TRAINING_CONFIG["max_tokens"],
    temperature=TRAINING_CONFIG["temperature"],
    lora_rank=TRAINING_CONFIG["lora_rank"],
    kl_penalty_coef=TRAINING_CONFIG["kl_penalty_coef"],
    num_substeps=TRAINING_CONFIG["num_substeps"],
    wandb_project=TRAINING_CONFIG["wandb_project"],
    wandb_name=run_name,
    log_path=log_path,
    eval_every=TRAINING_CONFIG["eval_every"],
    save_every=TRAINING_CONFIG["save_every"],
    stream_minibatch_config=None,
)

print("Training configuration created!")
print(f"Checkpoints will be saved to: {log_path}/checkpoints/")

Training configuration created!
Checkpoints will be saved to: /tmp/tinker-examples/mcq/mcq_Qwen-Qwen3-4B-Instruct-2507_lr1e-05_20260401-212914/checkpoints/


### Run Training

This will run the RL training loop. Monitor the output for:
- Average reward per step (should increase)
- Loss values
- KL divergence

**Warning**: Training can take significant time and compute resources. Start with a few steps to verify everything works.

In [11]:
# Run training (this will take a while!)
# For shorter testing, you can modify the config to run fewer steps
await train.main(training_cfg)

tinker_cookbook.utils.ml_log:475 [INFO] Logging to: /tmp/tinker-examples/mcq/mcq_Qwen-Qwen3-4B-Instruct-2507_lr1e-05_20260401-212914
tinker_cookbook.checkpoint_utils:21 [INFO] No checkpoints found at /tmp/tinker-examples/mcq/mcq_Qwen-Qwen3-4B-Instruct-2507_lr1e-05_20260401-212914/checkpoints.jsonl
tinker_cookbook.checkpoint_utils:52 [INFO] No checkpoints found with key state_path in /tmp/tinker-examples/mcq/mcq_Qwen-Qwen3-4B-Instruct-2507_lr1e-05_20260401-212914
tinker.lib.public_interfaces.service_client:75 [INFO] ServiceClient initialized for session 748944b8-4db8-589c-ba0b-743ee76b9640
tinker.lib.public_interfaces.service_client:159 [INFO] TrainingClient initialized for model 748944b8-4db8-589c-ba0b-743ee76b9640:train:0
tinker_cookbook.rl.train:1090 [INFO] Will train on 1 batches
tinker_cookbook.utils.misc_utils:20 [INFO] Starting save_checkpoint
tinker_cookbook.utils.misc_utils:23 [INFO] save_checkpoint took 1.07 seconds
tinker_cookbook.rl.train:134 [INFO] 
====== Trajectory Group 

tinker_cookbook.utils.ml_log:195 [INFO] 
                    Step 0                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                         ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ env/all/ac_tokens_per_turn     │ 7.000000  │
│ env/all/by_group/frac_all_bad  │ 0.080000  │
│ env/all/by_group/frac_all_good │ 0.920000  │
│ env/all/by_group/frac_mixed    │ 0.000000  │
│ env/all/evaluate_answer        │ 0.920000  │
│ env/all/num_turns              │ 1.000000  │
│ env/all/ob_tokens_per_turn     │ 88.320000 │
│ env/all/reward/total           │ 0.920000  │
│ env/all/total_ac_tokens        │ 1400      │
│ env/all/total_episodes         │ 200       │
│ env/all/total_ob_tokens        │ 17664     │
│ env/all/total_turns            │ 200       │
│ env/all/turns_per_episode      │ 1.000000  │
│ optim/entropy                  │ 0.000011  │
│ optim/kl_sample_train_v1       │ 0.000013  │
│ optim/kl_sample_train_v2       │ 0.000000  │
│ optim/lr         

## 6. Evaluate Trained Model

After training, let's evaluate the trained checkpoint to see improvement.

In [13]:
# Find the latest checkpoint
import glob
import json

checkpoints_file = f"{log_path}/checkpoints.jsonl"
if os.path.exists(checkpoints_file):
    with open(checkpoints_file, 'r') as f:
        checkpoints = [json.loads(line) for line in f]
    
    latest_checkpoint = checkpoints[-1]
    print("Checkpoint keys:", list(latest_checkpoint.keys()))
    checkpoint_path = latest_checkpoint['sampler_path']
    checkpoint_step = latest_checkpoint.get('step') or latest_checkpoint.get('batch') or latest_checkpoint.get('i_batch', len(checkpoints) - 1)
    
    print(f"Latest checkpoint: Step {checkpoint_step}")
    print(f"Path: {checkpoint_path}")
else:
    print("No checkpoints found. Make sure training completed successfully.")

Checkpoint keys: ['name', 'batch', 'state_path', 'sampler_path']
Latest checkpoint: Step 1
Path: tinker://748944b8-4db8-589c-ba0b-743ee76b9640:train:0/sampler_weights/final


In [14]:
# Load the trained checkpoint
trained_sampling = service.create_sampling_client(
    model_path=checkpoint_path,
    base_model=TRAINING_CONFIG["model_name"]
)

# Create new client with trained model, wrapped for verifiers compatibility
trained_tinker_client = TinkerAsyncOpenAIClient(trained_sampling, renderer, tokenizer)
trained_client = OpenAIChatCompletionsClient(trained_tinker_client)

print("Trained model loaded successfully!")

Trained model loaded successfully!


In [15]:
# Evaluate trained model
print("Evaluating trained model...\n")

start_time = time.time()
trained_results = env.evaluate_sync(
    client=trained_client,
    model="tinker",
    num_examples=10,
    rollouts_per_example=3,
    max_concurrent=16,
    sampling_args={
        "max_tokens": 50,
        "temperature": 0.7,
    }
)
end_time = time.time()

# Extract per-rollout data
trained_outputs = trained_results["outputs"]
trained_rewards = [o["reward"] for o in trained_outputs]

print(f"Evaluation completed in {end_time - start_time:.2f} seconds")
print("=" * 60)
print(f"Average Reward: {sum(trained_rewards) / len(trained_rewards):.3f}")
print(f"Std Dev: {np.std(trained_rewards):.3f}")
print(f"Total Evaluations: {len(trained_rewards)}")

# Compare with base model
base_avg = sum(rewards) / len(rewards)
trained_avg = sum(trained_rewards) / len(trained_rewards)
improvement = trained_avg - base_avg

print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"Base Model:    {base_avg:.3f}")
print(f"Trained Model: {trained_avg:.3f}")
print(f"Improvement:   {improvement:+.3f} ({improvement / base_avg * 100:+.1f}%)")

eval_dataset is not set, falling back to train dataset


Evaluating trained model...



Processing 10 groups (30 total rollouts): 100%|██████████| 10/10 [00:06<00:00,  1.46it/s, reward=0.9] 

Evaluation completed in 6.85 seconds
Average Reward: 0.900
Std Dev: 0.300
Total Evaluations: 30

COMPARISON
Base Model:    0.900
Trained Model: 0.900
Improvement:   +0.000 (+0.0%)


## 7. Analyze Training Results

Let's visualize the training progress.

In [16]:
import matplotlib.pyplot as plt

# Load training metrics from checkpoints.jsonl
steps = []
avg_rewards = []

with open(checkpoints_file, 'r') as f:
    for line in f:
        checkpoint = json.loads(line)
        steps.append(checkpoint['step'])
        # Assuming reward metrics are logged (adjust field name as needed)
        if 'metrics' in checkpoint and 'avg_reward' in checkpoint['metrics']:
            avg_rewards.append(checkpoint['metrics']['avg_reward'])

# Plot reward over time
if avg_rewards:
    plt.figure(figsize=(10, 6))
    plt.plot(steps, avg_rewards, marker='o', linewidth=2, markersize=6)
    plt.xlabel('Training Step', fontsize=12)
    plt.ylabel('Average Reward', fontsize=12)
    plt.title('Training Progress: MCQ Environment', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No reward metrics found in checkpoints. Metrics may be logged differently.")

KeyError: 'step'

## 8. Compare Specific Examples

Let's see side-by-side how the base model and trained model respond to the same questions.

In [ ]:
# Compare first 3 examples
for i in range(min(3, len(outputs))):
    base_o = outputs[i]
    trained_o = trained_outputs[i]
    
    print(f"\n{'=' * 80}")
    print(f"Question {i + 1}:")
    print("=" * 80)
    
    if base_o.get("prompt"):
        print("\n[PROMPT]")
        print(messages_to_printable(base_o["prompt"]))
    
    if base_o.get("completion"):
        print("\n[BASE MODEL RESPONSE]")
        print(messages_to_printable(base_o["completion"]))
    print(f"Reward: {base_o['reward']:.3f}")
    
    if trained_o.get("completion"):
        print("\n[TRAINED MODEL RESPONSE]")
        print(messages_to_printable(trained_o["completion"]))
    print(f"Reward: {trained_o['reward']:.3f}")
    
    if trained_o["reward"] > base_o["reward"]:
        print("\n✓ IMPROVED")
    elif trained_o["reward"] < base_o["reward"]:
        print("\n✗ REGRESSED")
    else:
        print("\n= SAME")

## 9. Test on Unseen Questions

To properly evaluate generalization, you should test on held-out questions not in the training set.

In [ ]:
# Create a simple test question
test_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that answers multiple choice questions.\n\nYou will be given a question and options. Respond with the correct option letter only (A, B, C, or D) in XML format:\n\n<answer>X</answer>\n\nReplace X with the correct option letter."
    },
    {
        "role": "user",
        "content": """What is 2 + 2?

A. 3
B. 4
C. 5
D. 6
"""
    }
]

# Test base model (use the underlying tinker client for direct chat completions)
print("Testing on new question: What is 2 + 2?")
print("\n[BASE MODEL]")
base_response = await tinker_client.chat.completions.create(
    model="tinker",
    messages=test_messages,
    max_tokens=50,
    temperature=0.7
)
print(base_response.choices[0].message.content)

# Test trained model
print("\n[TRAINED MODEL]")
trained_response = await trained_tinker_client.chat.completions.create(
    model="tinker",
    messages=test_messages,
    max_tokens=50,
    temperature=0.7
)
print(trained_response.choices[0].message.content)

## 10. Export and Use the Trained Model

The trained LoRA adapter is saved in the checkpoint directory. You can:

1. **Use with Tinker**: Load the checkpoint path directly
2. **Merge with base model**: Convert LoRA to full model weights
3. **Deploy**: Use in production with Tinker's serving infrastructure

Checkpoint location:

In [ ]:
print(f"Checkpoint directory: {checkpoint_path}")
print(f"\nTo use this checkpoint in future scripts:")
print(f"""\nservice = tinker.ServiceClient()
sampling = service.create_sampling_client(
    model_path="{checkpoint_path}",
    base_model="{TRAINING_CONFIG['model_name']}"
)""")

## Next Steps

### Improve the Environment

1. **Add more questions**: Expand the dataset for better generalization
2. **Add difficulty levels**: Weight harder questions higher
3. **Add reasoning**: Require model to explain its answer
4. **Multi-turn dialogue**: Allow model to ask clarifying questions

### Experiment with Hyperparameters

1. **Learning rate**: Try 5e-6, 1e-5, 5e-5
2. **LoRA rank**: Test 16, 32, 64
3. **Temperature**: Experiment with 0.5-1.5
4. **Batch size**: Adjust `group_size` and `groups_per_batch`
5. **KL penalty**: Add regularization with 0.01-0.1

### Deploy to Environments Hub

To share this environment:

1. Package as a Python module
2. Add `setup.py` or `pyproject.toml`
3. Upload to Prime Intellect Environments Hub
4. Others can install with: `prime env install yourusername/mcq-qa`

### Advanced Features

- **Chain of Thought**: Modify parser to extract reasoning steps
- **Partial Credit**: Reward based on reasoning quality even if final answer is wrong
- **Dynamic Difficulty**: Adjust question difficulty based on model performance
- **Multi-Task**: Combine with other environments for curriculum learning

## Appendix: Running Training from Command Line

Instead of running training in a notebook, you can use the command-line script:

```bash
# After making your environment loadable by vf.load_environment('mcq')
python -m tinker_cookbook.recipes.verifiers_rl.train \
    vf_env_id=mcq \
    learning_rate=1e-5 \
    groups_per_batch=32 \
    group_size=8
```

For evaluation:

```bash
python -m tinker_cookbook.recipes.verifiers_rl.evaluate \
    vf_env_id=mcq \
    model_path=/tmp/tinker-examples/mcq/run_name/checkpoints/step_50/sampler \
    num_examples=25 \
    rollouts_per_example=5
```